# 01 Data Merge and Preprocessing

Project: Predicting Thyroid Dysfunction Using Machine Learning

Author: Namitha Nair

This notebook is part of a cleaned research workflow for the NHANES thyroid-metabolic analysis.


## Purpose

This notebook loads the NHANES 2007–2008 files, merges them using `SEQN`, renames selected variables into readable names, and saves a clean processed dataset.

Expected raw files in `data/raw/`:

- `DEMO_E.xpt`
- `BMX_E.xpt`
- `GLU_E.xpt`
- `GHB_E.xpt`
- `THYROD_E.xpt`

`SEQN` is the NHANES respondent sequence number and is used to link records belonging to the same participant across NHANES component files.


In [ ]:
from pathlib import Path

# Project paths. These work if the notebooks are run from the repository root.
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FIGURES_DIR = Path("figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd

files = {
    "demo": "DEMO_E.xpt",
    "bmx": "BMX_E.xpt",
    "glu": "GLU_E.xpt",
    "ghb": "GHB_E.xpt",
    "thyroid": "THYROD_E.xpt"
}

# Read files from data/raw. If not found, try current working directory for Colab uploads.
def read_xpt(filename):
    path = RAW_DIR / filename
    if not path.exists():
        path = Path(filename)
    return pd.read_sas(path)

demo = read_xpt(files["demo"])
bmx = read_xpt(files["bmx"])
glu = read_xpt(files["glu"])
ghb = read_xpt(files["ghb"])
thyroid = read_xpt(files["thyroid"])

print("Raw dataset shapes:")
print("DEMO:", demo.shape)
print("BMX:", bmx.shape)
print("GLU:", glu.shape)
print("GHB:", ghb.shape)
print("THYROD:", thyroid.shape)


In [ ]:
# Merge all files on SEQN using inner joins so each remaining participant appears in every component file.
merged = demo.merge(bmx, on="SEQN", how="inner")
merged = merged.merge(glu, on="SEQN", how="inner")
merged = merged.merge(ghb, on="SEQN", how="inner")
merged = merged.merge(thyroid, on="SEQN", how="inner")

print("Merged shape:", merged.shape)
merged.head()


## Select variables

These variables were selected because they represent demographics, body composition, glucose/insulin metabolism, and thyroid function.


In [ ]:
selected_columns = [
    "SEQN",       # participant ID
    "RIDAGEYR",   # age
    "RIAGENDR",   # sex
    "RIDRETH1",   # race/ethnicity
    "BMXBMI",     # BMI
    "BMXWAIST",   # waist circumference
    "LBXGLU",     # fasting glucose
    "LBXIN",      # fasting insulin
    "LBXGH",      # HbA1c
    "LBXTSH1",    # TSH
    "LBXT3F",     # free T3
    "LBXT4F",     # free T4
    "LBXTPO",     # TPO antibodies
    "LBXATG"      # thyroglobulin antibodies
]

df = merged[selected_columns].copy()

df = df.rename(columns={
    "SEQN": "Participant_ID",
    "RIDAGEYR": "Age_Years",
    "RIAGENDR": "Sex",
    "RIDRETH1": "Race_Ethnicity",
    "BMXBMI": "BMI",
    "BMXWAIST": "Waist_Circumference_cm",
    "LBXGLU": "Fasting_Glucose_mg_dL",
    "LBXIN": "Fasting_Insulin_uU_mL",
    "LBXGH": "HbA1c_Percent",
    "LBXTSH1": "TSH",
    "LBXT3F": "Free_T3",
    "LBXT4F": "Free_T4",
    "LBXTPO": "TPO_Antibodies",
    "LBXATG": "Thyroglobulin_Antibodies"
})

# Convert NHANES sex codes to readable labels.
df["Sex"] = df["Sex"].map({1: "Male", 2: "Female"})

print("Number of participants after merge:", len(df))
df.head()


## Missingness check


In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2)
})
missing_summary


In [ ]:
# Save the merged dataset.
output_path = PROCESSED_DIR / "NHANES_Thyroid_Glucose_Dataset.csv"
df.to_csv(output_path, index=False)
print(f"Saved processed dataset to: {output_path}")
